# 🫀 AI랩 Week 02 — 12유도 ECG 다중라벨 진단 (PTB-XL · 1D-ResNet)

**목표(게이트)**: 상위 5진단군(NORM·MI·STTC·CD·HYP) **macro-AUROC ≥ 0.85**

1주차(MIT-BIH·1D-CNN·단일 비트, softmax)에서 → **12유도 10초 한 장·1D-ResNet·다중라벨(sigmoid)** 로 확장한다.
카드: `content/ailab/ailab-2026-0012.md`(실습) · `ailab-2026-0013.md`(원본 코드 분석) · `ailab-2026-0014.md`(심화 퀘스트)

> Colab에서 **런타임 → GPU(T4)** 로 설정하고 위에서부터 실행하세요. PTB-XL 다운로드(100Hz)는 몇 분 걸립니다.

In [ ]:
# CELL 1 — 패키지 설치 (Colab 최초 1회)
!pip -q install wfdb wget
import torch
print('✅ torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# CELL 2 — 설정 & 임포트
import os, ast, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_fscore_support

# ---- 설정 (여기만 바꾸면 됨) ----
SR         = 100          # 100Hz(records100, 12x1000) — 입문·Colab용. 500도 가능
EPOCHS     = 20
BATCH      = 128
DEV_MODE   = False        # True면 소량 fold만 써서 빠르게 스모크 테스트
CLASSES    = ['NORM','MI','STTC','CD','HYP']   # SCP superdiagnostic 상위 5군
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(42); np.random.seed(42)
print('설정 완료 | SR =', SR, 'Hz | device =', DEVICE)

In [ ]:
# CELL 3 — PTB-XL 다운로드 + 라벨(superdiagnostic) 구성 (PhysioNet, 오픈·CC BY 4.0)
PTBXL = 'ptbxl'
if not os.path.exists(PTBXL):
    url = ('https://physionet.org/static/published-projects/ptb-xl/'
           'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip')
    print('▶ PTB-XL 내려받는 중(수 분)…')
    !wget -q -O ptbxl.zip {url}
    !unzip -q ptbxl.zip
    !mv ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 {PTBXL}
print('✅ 데이터 준비:', PTBXL)

# 메타·라벨: ptbxl_database.csv 의 scp_codes 를 scp_statements.csv 로 상위군(diagnostic_class)에 매핑
df = pd.read_csv(f'{PTBXL}/ptbxl_database.csv', index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(ast.literal_eval)
agg = pd.read_csv(f'{PTBXL}/scp_statements.csv', index_col=0)
agg = agg[agg.diagnostic == 1]

def to_superclasses(codes):
    out = set()
    for c in codes:                       # codes: {SCP코드: likelihood}
        if c in agg.index:
            out.add(agg.loc[c, 'diagnostic_class'])
    return list(out)

df['super'] = df.scp_codes.apply(to_superclasses)
# 5차원 멀티핫 라벨(한 심전도에 여러 진단 동시 참 → 다중라벨)
C2I = {c:i for i,c in enumerate(CLASSES)}
Y = np.zeros((len(df), len(CLASSES)), dtype='float32')
for r,(_,supers) in enumerate(df['super'].items()):
    for s in supers:
        if s in C2I: Y[r, C2I[s]] = 1.0
df['_row'] = np.arange(len(df))
print('기록 수:', len(df), '| 라벨 분포(양성 수):', dict(zip(CLASSES, Y.sum(0).astype(int))))

In [ ]:
# CELL 4 — 신호 로드 + PTB-XL 표준 분할(strat_fold: 1-8 train / 9 val / 10 test)
import wfdb
col = 'filename_lr' if SR == 100 else 'filename_hr'

def load_signals(frame):
    sigs = []
    for fn in frame[col]:
        rec, _ = wfdb.rdsamp(f'{PTBXL}/{fn}')     # (samples, 12)
        sigs.append(rec.T.astype('float32'))       # → (12, samples)
    X = np.stack(sigs)
    X = (X - X.mean(axis=(0,2), keepdims=True)) / (X.std(axis=(0,2), keepdims=True) + 1e-8)
    return X

folds = df.strat_fold.values
tr_m, va_m, te_m = folds <= 8, folds == 9, folds == 10
if DEV_MODE:                                        # 스모크: 각 split 앞부분만
    def cap(mask, n):
        idx = np.where(mask)[0][:n]; m = np.zeros_like(mask); m[idx] = True; return m
    tr_m, va_m, te_m = cap(tr_m, 800), cap(va_m, 200), cap(te_m, 200)

print('신호 로딩 중… (train)'); Xtr = load_signals(df[tr_m])
print('신호 로딩 중… (val)');   Xva = load_signals(df[va_m])
print('신호 로딩 중… (test)');  Xte = load_signals(df[te_m])
ytr, yva, yte = Y[tr_m], Y[va_m], Y[te_m]
print('train', Xtr.shape, '| val', Xva.shape, '| test', Xte.shape)

def loader(X, y, shuffle):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle)
dl_tr, dl_va, dl_te = loader(Xtr,ytr,True), loader(Xva,yva,False), loader(Xte,yte,False)

In [ ]:
# CELL 5 — 1D-ResNet (PyTorch) — 잔차 블록이 2주차의 핵심
class BasicBlock1d(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 5, stride, padding=2, bias=False)
        self.bn1   = nn.BatchNorm1d(cout)
        self.conv2 = nn.Conv1d(cout, cout, 3, 1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(cout)
        self.down  = (nn.Sequential(nn.Conv1d(cin, cout, 1, stride, bias=False),
                                    nn.BatchNorm1d(cout))
                      if stride != 1 or cin != cout else None)
    def forward(self, x):
        idt = x if self.down is None else self.down(x)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return torch.relu(x + idt)                 # F(x) + x  ← 잔차 지름길

class ResNet1d(nn.Module):
    def __init__(self, n_classes=5, in_ch=12):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv1d(in_ch, 64, 7, 2, 3, bias=False),
                                    nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2))
        self.layer1 = nn.Sequential(BasicBlock1d(64, 64),   BasicBlock1d(64, 64))
        self.layer2 = nn.Sequential(BasicBlock1d(64, 128, 2), BasicBlock1d(128, 128))
        self.layer3 = nn.Sequential(BasicBlock1d(128, 256, 2), BasicBlock1d(256, 256))
        self.head   = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                    nn.Linear(256, n_classes))   # logits (sigmoid는 손실에서)
    def forward(self, x):
        return self.head(self.layer3(self.layer2(self.layer1(self.stem(x)))))

model = ResNet1d(len(CLASSES), in_ch=12).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model); print('파라미터 수:', f'{n_params:,}')

In [ ]:
# CELL 6 — 학습 (다중라벨: BCEWithLogitsLoss, 베스트는 val macro-AUROC 기준)
loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

@torch.no_grad()
def predict(dl):
    model.eval(); ps, ys = [], []
    for xb, yb in dl:
        ps.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy()); ys.append(yb.numpy())
    return np.concatenate(ps), np.concatenate(ys)

def macro_auroc(y, p):
    cols = [i for i in range(y.shape[1]) if y[:,i].sum() > 0]   # 양성 없는 열 제외
    return roc_auc_score(y[:,cols], p[:,cols], average='macro')

best, best_state = 0.0, None
for ep in range(1, EPOCHS+1):
    model.train()
    for xb, yb in dl_tr:
        opt.zero_grad()
        loss = loss_fn(model(xb.to(DEVICE)), yb.to(DEVICE))
        loss.backward(); opt.step()
    pv, yv = predict(dl_va); auc = macro_auroc(yv, pv)
    if auc > best:
        best, best_state = auc, {k: v.cpu().clone() for k,v in model.state_dict().items()}
    print(f'epoch {ep:2d} | val macro-AUROC = {auc:.4f}  (best {best:.4f})')
if best_state: model.load_state_dict(best_state)
print('✅ 학습 완료 · best val macro-AUROC =', round(best,4))

In [ ]:
# CELL 7 — 평가: test macro-AUROC + 클래스별 AUROC 표 + 통과 판정
pt, yt = predict(dl_te)
test_auroc = macro_auroc(yt, pt)
print('='*52)
print(f'  test macro-AUROC = {test_auroc:.4f}   →   '
      f"{'✅ PASS (≥0.85)' if test_auroc>=0.85 else '❌ 미달 (<0.85)'}")
print('='*52)

# 클래스별 AUROC + (검증셋 F1-최대 임계값에서의) precision/recall/F1  ← 게이트 산출물
print(f"{'진단군':<6} {'AUROC':>7} {'thr':>6} {'P':>6} {'R':>6} {'F1':>6}  {'양성수':>6}")
pv, yv = predict(dl_va)
for i, c in enumerate(CLASSES):
    if yt[:,i].sum() == 0: 
        print(f'{c:<6}   (test 양성 없음)'); continue
    auc_i = roc_auc_score(yt[:,i], pt[:,i])
    ths = np.linspace(0.05, 0.95, 19)
    f1s = [f1_score(yv[:,i], (pv[:,i]>=t).astype(int), zero_division=0) for t in ths]
    thr = ths[int(np.argmax(f1s))]
    p,r,f,_ = precision_recall_fscore_support(yt[:,i], (pt[:,i]>=thr).astype(int),
                                              average='binary', zero_division=0)
    print(f'{c:<6} {auc_i:7.3f} {thr:6.2f} {p:6.3f} {r:6.3f} {f:6.3f}  {int(yt[:,i].sum()):>6}')

In [ ]:
# CELL 8 — (선택) 원본 오픈소스 벤치마크 코드 받아서 나란히 읽기
# GPL-3.0 저장소라 '받아서 읽기'만 한다(복사·커밋 금지). MedKOS fetch_project.py 와 같은 동작.
import os
if not os.path.exists('ecg_ptbxl_benchmarking'):
    !git clone --depth 1 https://github.com/helme/ecg_ptbxl_benchmarking.git
print('\n▶ 원본 resnet1d.py 첫 부분 (우리 ResNet1d 와 대조):')
!sed -n '1,60p' ecg_ptbxl_benchmarking/code/models/resnet1d.py
# 논문 재현 전체 실험: ecg_ptbxl_benchmarking/code/reproduce_results.py (데이터는 get_datasets.sh)

In [ ]:
# CELL 9 — Google Drive에 체크포인트 + 결과 저장 (check_week.py 가 읽는 형식)
from google.colab import drive
import json, datetime
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/MedKOS/ailab/week02'
os.makedirs(OUT, exist_ok=True)
torch.save(model.state_dict(), f'{OUT}/week02_best.pt')

result = {
    'week': 2, 'task': '12-lead ECG multi-label (PTB-XL superdiagnostic)',
    'metric': 'macro_auroc', 'value': round(float(test_auroc), 4),
    'passed': bool(test_auroc >= 0.85),
    'classes': CLASSES, 'sr': SR,
    'date': datetime.date.today().isoformat(),
}
with open(f'{OUT}/week02_result.json', 'w') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print('✅ Drive 저장 완료:', OUT)
print(json.dumps(result, ensure_ascii=False, indent=2))

## MedKOS에 기록하기 (마지막 단계)

1. `week02_result.json` 을 로컬로 내려받아 판정·진급:
   ```bash
   python pipelines/check_week.py --results week02_result.json   # 통과 시 자동으로 3주차로
   ```
2. **실행 로그로 박기**(예측 방지): 이 노트북을 `notebooks/` 에 커밋하고
   ```bash
   python pipelines/ingest_run.py --results week02_result.json --notebook notebooks/week02_ptbxl_resnet1d.ipynb
   ```
3. 관찰(약한 진단군·임계값 등)을 카드 `ailab-2026-0012.md` 의 `## My notes` 에 한 줄.
   더 파볼 열린 문제는 심화 퀘스트 `ailab-2026-0014.md` 로 이어진다.